# Nusselt number: LBM vs. literature

Compare time-averaged heat transfer from the CPU LBM solver with the square-enclosure Rayleigh–Bénard benchmark of Ouertatani et al. The reference points used here are the same values used by this repository's `validate_transision` routine.

**Reference:** N. Ouertatani, N. Ben Cheikh, B. Ben Beya, and T. Lili, “Numerical simulation of two-dimensional Rayleigh–Bénard convection in an enclosure,” *Comptes Rendus Mécanique*, **336** (5), 464–470 (2008). [doi:10.1016/j.crme.2008.02.004](https://doi.org/10.1016/j.crme.2008.02.004). The article presents benchmark solutions over Rayleigh numbers from $10^3$ to $10^6$.

| Ra | Reference Nu |
|---:|---:|
| $10^3$ | 1.0004 |
| $10^4$ | 2.1580 |
| $10^5$ | 3.9103 |
| $10^6$ | 6.3092 |

The comparison uses a two-dimensional 1:1 domain (`NX = NY`), `Pr = 0.71`, constant hot/cold horizontal plates, and `wallprop="WALLED"` (no-slip side walls). The plotted error bars show the standard deviation across time samples; they do not measure grid-discretization error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from controlRB import lbmd2q9

reference_ra = np.array([1e3, 1e4, 1e5, 1e6], dtype=float)
reference_nu = np.array([1.0004, 2.1580, 3.9103, 6.3092], dtype=float)

## Run settings

The default run does a 20,000-step transient, then takes 20 samples separated by 2,000 steps for each Rayleigh number. All step counts are even as required by the solver. Increase the resolution and sampling duration and compare successive runs before treating agreement as a convergence result. The full sweep can take a few minutes on CPU.

In [ ]:
N = 128
Pr = 0.71
warmup_steps = 50_000 * (N//64)**2
sample_interval = 2_000 * (N//64)**2
n_samples = 100
rayleigh_numbers = reference_ra.copy()

assert warmup_steps % 2 == 0
assert sample_interval % 2 == 0
assert n_samples > 0
print(f"Square grid: {N} x {N}; total sampled steps per Ra: {n_samples * sample_interval:,}")

In [ ]:
results = []

for Ra in rayleigh_numbers:
    lbm = lbmd2q9(
        mode="cuda",
        NX=N,
        NY=N,
        NSIM=1,
        Ra=float(Ra),
        Pr=Pr,
        TBOT0=2.0,
        TTOP0=1.0,
        wallprop="WALLED",
        verbose=False,
        name=f"nu_ref_{int(Ra)}",
    )
    try:
        lbm.allsteps(nsteps=warmup_steps)
        nu_samples = []
        for _ in range(n_samples):
            lbm.allsteps(nsteps=sample_interval)
            nu_samples.append(float(lbm.calNu()))

        reference = reference_nu[np.flatnonzero(reference_ra == Ra)[0]]
        mean_nu = float(np.mean(nu_samples))
        results.append({
            "Ra": float(Ra),
            "Nu_mean": mean_nu,
            "Nu_std": float(np.std(nu_samples, ddof=1)) if n_samples > 1 else 0.0,
            "Nu_reference": float(reference),
            "relative_error_percent": 100.0 * (mean_nu - reference) / reference,
            "time_steps": int(lbm.t[0]),
            "nu_samples": nu_samples,
        })
        print(f"Ra={Ra:.0e}: Nu={mean_nu:.5f} +/- {results[-1]['Nu_std']:.5f} (reference {reference:.5f})")
    finally:
        lbm.close()


In [ ]:
print(f"{'Ra':>10} {'LBM Nu':>12} {'time std':>12} {'reference Nu':>14} {'relative error':>16}")
print("-" * 68)
for row in results:
    print(
        f"{row['Ra']:10.0e} {row['Nu_mean']:12.5f} {row['Nu_std']:12.5f} "
        f"{row['Nu_reference']:14.5f} {row['relative_error_percent']:15.2f}%"
    )

In [ ]:
simulation_ra = np.array([row["Ra"] for row in results])
simulation_nu = np.array([row["Nu_mean"] for row in results])
simulation_std = np.array([row["Nu_std"] for row in results])

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(
    simulation_ra,
    simulation_nu,
    yerr=simulation_std,
    fmt="o-",
    capsize=4,
    label=f"LBM, {N} x {N}",
)
ax.plot(reference_ra, reference_nu, "s--", label="Ouertatani et al. (2008)")
ax.set_xscale("log")
ax.set_xlabel("Rayleigh number, Ra")
ax.set_ylabel("Nusselt number, Nu")
ax.set_title("Square no-slip Rayleigh–Bénard enclosure")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Interpreting the comparison

A discrepancy can come from transient sampling, finite grid resolution (especially near thermal and velocity boundary layers at high $Ra$), or differences in numerical/model details. Repeat with a larger `N` and longer `warmup_steps`/sampling, then compare the means across resolutions. The reported sample standard deviation only describes variation over the selected time samples; it is not an uncertainty estimate for the literature values.